# 1R1C driven by the measured evaporator temperature (COP-free) — heat balances and state-space matrices

```
   T_a ---R_ia--- T_i ---g(s)--- T_ev (measured)     g(s) = s/R_ie_on + (1-s)/R_ie_off
                   |
                  C_i
```

| symbol | unit | meaning |
|---|---|---|
| $T_i$ | °C | freezer air (state, measured, thermostat sensor) |
| $T_a$ | °C | neighbouring cold room, `Cold room temperature` (input) |
| $T_{ev}$ | °C | measured evaporator temperature, `Evaporator temperature` (input) |
| $s$ | – | compressor/fan on-fraction of the interval (input, `on`) |

Free parameters: `C_i`, `R_ia`, `R_ie_on`, `R_ie_off`. Registry key `fz_1R1C_evin_temperature`, class
`Freezer1R1C_EVIN` (a fixed configuration of `FreezerNetwork`, see `freezer_models/network.py`).
The air–coil coupling is fan-gated: $g(s) = s/R_{ie}^{on} + (1-s)/R_{ie}^{off}$ with $s$ the compressor on-fraction, so the matrices depend on $s$ (switched linear model, affine in $s$).

The cooling is driven by the measured coil temperature, so the COP does not enter.


In [1]:
import numpy as np
import sympy as sp
from IPython.display import display, Math

sp.init_printing()

def show(lhs, expr):
    display(Math(lhs + " = " + sp.latex(expr)))

## Heat balances

$$C_{i} \frac{dT_i}{dt} = \frac{T_{a} - T_{i}}{R_{ia}} + \left(- T_{i} + T_{ev}\right) \left(\frac{s}{R_{ie}^{on}} + \frac{1 - s}{R_{ie}^{off}}\right)$$

In [2]:
T_i, T_m, T_e, T_a, T_out, T_ev, Phi_HC = sp.symbols("T_i T_m T_e T_a T_out T_ev Phi_HC", real=True)
R_ia, R_im, R_ie_on, R_ie_off, R_ea, R_iout = sp.symbols("R_ia R_im R_ie_on R_ie_off R_ea R_iout", positive=True)
C_i, C_m, C_e = sp.symbols("C_i C_m C_e", positive=True)
beta = sp.Symbol("beta", real=True)
s = sp.Symbol("s", nonnegative=True)   # compressor on-fraction
dT_i, = sp.symbols(r"\dot{T_i}", seq=True)

balances = [
    sp.Eq(C_i * dT_i, (T_a - T_i) / R_ia + (s / R_ie_on + (1 - s) / R_ie_off) * (T_ev - T_i)),
]
for b in balances:
    display(b)

sol = sp.solve(balances, [dT_i], dict=True)[0]
f = sp.Matrix([sp.expand(sol[dT_i])])
for name, fx in zip(['T_i'], f):
    show(r"\dot{" + name + "}", fx)

<IPython.core.display.Math object>

## State-space form

The balances are linear in the states and in the input vector

$$u = [T_{a}, T_{ev}]$$

(the order of `INPUT_NAMES` in the class), so $\dot x = A(s)\,x + B(s)\,u$ with $A = \partial f/\partial x$ and $B = \partial f/\partial u$.

In [3]:
x = sp.Matrix([T_i])
w = sp.Symbol("w")   # stands for the product input Phi_HC*(T_out - 10)
u_expr = [T_a, T_ev]
u_sym = [T_a, T_ev]
f_w = f.subs(Phi_HC * beta * T_out, beta * (w + 10 * Phi_HC)).expand() if False else f
f_w = sp.expand(f_w)
A = f_w.jacobian(x)
B = f_w.jacobian(sp.Matrix(u_sym))
show("A", sp.simplify(A))
show("B", sp.simplify(B))
residual = sp.simplify((A * x + B * sp.Matrix(u_sym) - f_w))
assert residual == sp.zeros(*f.shape), residual
print("A x + B u == f  ✓  (u =", u_sym, ")")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

A x + B u == f  ✓  (u = [T_a, T_ev] )


## Check against the toolbox implementation

The symbolic matrices are evaluated for random parameter values (and on/off states) and compared with
`calc_matrices(s)` of the class. The cells after that check the discretization and the limits numerically:

- exact ZOH (`zoh_discretize`, used for identification) against a forward-Euler simulation with a 1000× finer step;
- the steady state of a long simulation with constant inputs against $x_{ss} = -A^{-1}B u$;
- reduction to the parent model when the extra element is removed.

In [4]:
from rctoolbox.models.rc.freezer_models.fz_1r1c_evin.model import Freezer1R1C_EVIN
from rctoolbox.models.rc.freezer_models._checks import random_params

rng = np.random.default_rng(0)
symmap = {"R_ia": R_ia, "R_im": R_im, "R_ie_on": R_ie_on, "R_ie_off": R_ie_off, "R_ea": R_ea, "R_iout": R_iout,
          "C_i": C_i, "C_m": C_m, "C_e": C_e, "beta": beta}
model = Freezer1R1C_EVIN()
print("states", model.STATE_NAMES, "| inputs", model.INPUT_NAMES, "| switched", model.SWITCHED)
for _ in range(5):
    p = random_params(model, rng)
    model.set_params(p)
    for sv in (0.0, 0.5, 1.0):
        A_num, B_num = model.calc_matrices(sv)
        subs = {symmap[k]: v for k, v in p.items()} | {s: sv}
        assert np.allclose(np.array(A.subs(subs), float), A_num, rtol=1e-10, atol=1e-15)
        assert np.allclose(np.array(B.subs(subs), float), B_num, rtol=1e-10, atol=1e-15)
print("symbolic A, B == Freezer1R1C_EVIN.calc_matrices(s)  ✓")

states ('T_i',) | inputs ('T_a', 'T_ev') | switched True


symbolic A, B == Freezer1R1C_EVIN.calc_matrices(s)  ✓


In [5]:
from rctoolbox.models.rc.freezer_models._checks import check_zoh_vs_euler, check_steady_state, check_reduction

print("max |ZOH - fine Euler| over 2 h (K):", check_zoh_vs_euler(model, p))
print("max |steady state - (-A^-1 B u)| (K):", check_steady_state(model, p))

max |ZOH - fine Euler| over 2 h (K): 0.00028329168023333295
max |steady state - (-A^-1 B u)| (K): 3.219930455600206e-08
